<a href="https://colab.research.google.com/github/Khadijabenabdelkader/akt26_gpu_dev/blob/main/gemm_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
!nvidia-smi

Fri Oct 30 17:22:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [57]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [58]:
!git clone https://github.com/Khadijabenabdelkader/akt26_gpu_dev.git

fatal: destination path 'akt26_gpu_dev' already exists and is not an empty directory.


In [59]:
%cd akt26_gpu_dev/3-ger

/content/akt26_gpu_dev/4-gemm/akt26_gpu_dev/3-ger


In [60]:
!ls

ger  ger.cu  makefile


In [61]:
#Compilez et lancez
!nvcc ger.cu -o ger -lcublas
!./ger

Matrix: 4000 x 2000, alpha = 1.50, repeats = 100
Matrix storage: column-major

Implementation          Time (us)      GFLOP/s           Error Norm
--------------------------------------------------------------------
Custom kernel             289.333       55.300         4.440474e-08
cuBLAS SGER               314.367       50.896         4.440474e-08

TEST PASS
Speedup (cuBLAS time / custom time): 1.087x


In [62]:
!git status
!git add 3-ger
!git commit -m "Complete assignment 1: GER kernel"
!git push

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	ger

nothing added to commit but untracked files present (use "git add" to track)
fatal: pathspec '3-ger' did not match any files
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@b6d258ad8e3d.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [77]:
%%bash
cd /content/akt26_gpu_dev

git pull origin main

Updating 70c29e2..08c62a5
Fast-forward
 4-gemm/gemm.cu | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


From https://github.com/Khadijabenabdelkader/akt26_gpu_dev
 * branch            main       -> FETCH_HEAD
   70c29e2..08c62a5  main       -> origin/main


In [78]:
%cd /content/akt26_gpu_dev/4-gemm

!pwd
!ls -l gemm.cu
!grep -n "DEBUG INFORMATION" gemm.cu
!grep -n "FIRST 10 RESULTS" gemm.cu

/content/akt26_gpu_dev/4-gemm
/content/akt26_gpu_dev/4-gemm
-rw-r--r-- 1 root root 16216 Oct 30 17:31 gemm.cu
182:    printf("                 DEBUG INFORMATION\n");
189:    printf("\n--- FIRST 10 RESULTS (BATCH 0) ---\n");


In [79]:
%cd /content/akt26_gpu_dev/4-gemm

!grep -n "__global__ void batch_gemm_kernel" gemm.cu
!head -n 40 gemm.cu

/content/akt26_gpu_dev/4-gemm
41:__global__ void batch_gemm_kernel(
#include <cuda_runtime.h>
#include <cublas_v2.h>

#include <algorithm>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <vector>

// ============================================================
// Vérification des erreurs CUDA
// ============================================================
#define CUDA_CHECK(call)                                                   \
    do {                                                                   \
        cudaError_t err = (call);                                          \
        if (err != cudaSuccess) {                                          \
            fprintf(stderr, "CUDA error at %s:%d: %s\n",                   \
                    __FILE__, __LINE__, cudaGetErrorString(err));          \
            exit(EXIT_FAILURE);                                            \
        }                                                                  \
    } while

In [80]:
!nvcc -O2 gemm.cu -lcublas -o gemm

In [82]:
!./gemm 8 100
!./gemm 16 1000
!./gemm 32 1000

GPU: Tesla T4 (compute capability 7.5)
Matrix size: 8 x 8, batch size: 100
FP32, column-major matrices, repeats: 100


                 DEBUG INFORMATION

--- INPUT DATA ---
A[0] = 0.012999634
B[0] = 0.999855518

--- FIRST 10 RESULTS (BATCH 0) ---
Index    CPU              Custom CUDA      cuBLAS          
0        2.846466303      2.846466303      2.846466303     
1        2.939879417      2.939879417      2.939879417     
2        3.032795429      3.032795429      3.032795429     
3        3.125199080      3.125199080      3.125199318     
4        3.217074633      3.217074633      3.217074394     
5        3.308406353      3.308406353      3.308406353     
6        3.399179220      3.399178982      3.399178982     
7        3.489377022      3.489377260      3.489377022     
8        2.781373024      2.781373024      2.781373262     
9        2.873001337      2.873001337      2.873001099     

--- FIRST RESULT OF LAST BATCH ---
Batch index       : 99
CPU reference     : 3.785136223
C